# 0825_peace_005_type_expert_fold_ensemble

타입별 XGBoost 전문가 모델에 시간순 Fold 앙상블을 실제 학습·추론 방식으로 적용한 실험입니다.

- 누적 체크포인트 0~30%, 0~40%, 0~50%, 0~70%에서 각각 타입별 모델 5개를 독립 학습합니다.
- Walk-forward에서는 해당 시점까지 존재하는 체크포인트 모델의 확률을 동일 가중 평균합니다.
- 최종 Validation/Test는 네 체크포인트 모델의 평균 확률로 평가합니다.
- 분할·피처·파라미터·평가 지표는 `0825_peace_004_type_expert_walk_forward`와 동일합니다.


## 1. 설정, 경로 탐색과 실행 로그

In [1]:
import gc
import hashlib
import json
import logging
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import xgboost
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

EXPERIMENT_ID = "0825_peace_005_type_expert_fold_ensemble"
RANDOM_STATE = 42
TARGET = "class"
TIME_COLUMN = "timestamp"
TYPE_COLUMN = "inspection_type"
RECORD_ID = "record_id"
DECISION_THRESHOLD = 0.5
MIN_RECALL = 0.99
TRAIN_END_FRACTION = 0.70
VALIDATION_END_FRACTION = 0.80

XGB_PARAMS = {
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "tree_method": "hist",
    "n_estimators": 400,
    "learning_rate": 0.05,
    "max_depth": 5,
    "min_child_weight": 10,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 5.0,
    "max_delta_step": 1.0,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbosity": 0,
}


def find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "notebooks").is_dir():
            return candidate.resolve()
    raise FileNotFoundError("AGENTS.md가 있는 저장소 루트를 찾지 못했습니다.")


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def find_data_pair(repo_root: Path) -> tuple[Path, Path]:
    candidates = [
        repo_root / "data" / "raw",
        repo_root.parent,
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent,
    ]
    checked = set()
    for directory in candidates:
        resolved = directory.resolve()
        if resolved in checked:
            continue
        checked.add(resolved)
        data_path = resolved / "dataset.csv"
        mapping_path = resolved / "mapping.json"
        if data_path.exists() and mapping_path.exists():
            return data_path, mapping_path
    raise FileNotFoundError("dataset.csv와 mapping.json 쌍을 찾지 못했습니다.")


REPO_ROOT = find_repo_root()
DATA_PATH, MAPPING_PATH = find_data_pair(REPO_ROOT)
LOG_DIR = REPO_ROOT / "docs" / "peace"
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = LOG_DIR / f"{EXPERIMENT_ID}.log"

logger = logging.getLogger(EXPERIMENT_ID)
logger.setLevel(logging.INFO)
logger.handlers.clear()
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
file_handler = logging.FileHandler(LOG_PATH, mode="w", encoding="utf-8")
file_handler.setFormatter(formatter)
stream_handler = logging.StreamHandler(sys.stdout)
stream_handler.setFormatter(formatter)
logger.addHandler(file_handler)
logger.addHandler(stream_handler)
logger.propagate = False

DATA_SHA256_BEFORE = sha256_file(DATA_PATH)
MAPPING_SHA256_BEFORE = sha256_file(MAPPING_PATH)
logger.info("experiment=%s", EXPERIMENT_ID)
logger.info(
    "random_state=%d baseline_threshold=%.2f min_recall=%.2f",
    RANDOM_STATE,
    DECISION_THRESHOLD,
    MIN_RECALL,
)
logger.info("data_file=%s sha256=%s", DATA_PATH.name, DATA_SHA256_BEFORE)
logger.info("mapping_file=%s sha256=%s", MAPPING_PATH.name, MAPPING_SHA256_BEFORE)
logger.info(
    "versions python=%s pandas=%s sklearn=%s xgboost=%s",
    sys.version.split()[0], pd.__version__, sklearn.__version__, xgboost.__version__
)
logger.info("log_file=docs/peace/%s", LOG_PATH.name)
print("log saved to:", LOG_PATH.relative_to(REPO_ROOT))


2026-08-25 15:34:20,425 | INFO | experiment=0825_peace_005_type_expert_fold_ensemble


2026-08-25 15:34:20,425 | INFO | random_state=42 baseline_threshold=0.50 min_recall=0.99


2026-08-25 15:34:20,426 | INFO | data_file=dataset.csv sha256=53e8568743216d556856ed69b388f6750fbfa0b8c59ad31f970515ac9eb10e62


2026-08-25 15:34:20,426 | INFO | mapping_file=mapping.json sha256=3b20f440b6d9ed0baefa662e1a6f03688befbe0f28341a3b54655d3058c6e486


2026-08-25 15:34:20,427 | INFO | versions python=3.12.7 pandas=2.2.2 sklearn=1.5.1 xgboost=3.4.1


2026-08-25 15:34:20,427 | INFO | log_file=docs/peace/0825_peace_005_type_expert_fold_ensemble.log


log saved to: docs/peace/0825_peace_005_type_expert_fold_ensemble.log


## 2. 원본 데이터와 매핑 검증

In [2]:
raw_df = pd.read_csv(DATA_PATH, low_memory=False)
source_index_column = raw_df.columns[0]
if source_index_column.startswith("Unnamed:") or source_index_column == "":
    raw_df = raw_df.rename(columns={source_index_column: RECORD_ID})
elif source_index_column != RECORD_ID:
    raise ValueError(f"예상하지 못한 첫 번째 컬럼: {source_index_column}")

with MAPPING_PATH.open(encoding="utf-8") as stream:
    feature_mapping = json.load(stream)

required_columns = {RECORD_ID, TIME_COLUMN, TYPE_COLUMN, TARGET}
missing_required = required_columns - set(raw_df.columns)
assert not missing_required, f"필수 컬럼 누락: {sorted(missing_required)}"
assert len(raw_df) == 440_274
assert raw_df[RECORD_ID].is_unique
assert set(raw_df[TARGET].unique()) == {0, 1}
assert raw_df[TARGET].value_counts().to_dict() == {0: 435_652, 1: 4_622}
assert set(raw_df[TYPE_COLUMN].unique()) == {0, 1, 2, 3, 4}
assert set(feature_mapping) == {"0", "1", "2", "3", "4"}

raw_df[TIME_COLUMN] = pd.to_datetime(raw_df[TIME_COLUMN], errors="raise", utc=True)
raw_df = raw_df.sort_values([TIME_COLUMN, RECORD_ID], kind="stable").reset_index(drop=True)
inspection_columns = [column for column in raw_df.columns if column.startswith("inspection_feat")]
mapped_union = set().union(*(set(columns) for columns in feature_mapping.values()))
assert len(inspection_columns) == 70
assert len(mapped_union) == 65
assert mapped_union <= set(inspection_columns)

numeric_inputs = raw_df.select_dtypes(include=[np.number]).drop(columns=[TARGET, RECORD_ID])
assert np.isfinite(numeric_inputs.to_numpy()).all()

data_summary = pd.Series(
    {
        "rows": len(raw_df),
        "columns": raw_df.shape[1],
        "false_call_0": int((raw_df[TARGET] == 0).sum()),
        "real_defect_1": int((raw_df[TARGET] == 1).sum()),
        "real_defect_rate_pct": raw_df[TARGET].mean() * 100,
        "inspection_types": raw_df[TYPE_COLUMN].nunique(),
        "inspection_features": len(inspection_columns),
        "mapped_feature_union": len(mapped_union),
        "timestamp_start": raw_df[TIME_COLUMN].min(),
        "timestamp_end": raw_df[TIME_COLUMN].max(),
    },
    name="raw_data",
)
display(data_summary)
logger.info(
    "data_verified rows=%d columns=%d class_0=%d class_1=%d",
    len(raw_df), raw_df.shape[1], int((raw_df[TARGET] == 0).sum()), int((raw_df[TARGET] == 1).sum())
)


rows                                       440274
columns                                        78
false_call_0                               435652
real_defect_1                                4622
real_defect_rate_pct                     1.049801
inspection_types                                5
inspection_features                            70
mapped_feature_union                           65
timestamp_start         1970-06-23 03:58:55+00:00
timestamp_end           1970-11-02 14:21:28+00:00
Name: raw_data, dtype: object

2026-08-25 15:34:24,799 | INFO | data_verified rows=440274 columns=78 class_0=435652 class_1=4622


## 3. 타입별 유효 피처

In [3]:
inspection_types = sorted(raw_df[TYPE_COLUMN].unique().tolist())
meta_columns = [column for column in raw_df.columns if column.startswith("meta_feat")]
feature_columns_by_type = {}
feature_rows = []

for inspection_type in inspection_types:
    mapped_columns = feature_mapping[str(inspection_type)]
    assert len(mapped_columns) == len(set(mapped_columns))
    assert set(mapped_columns) <= set(raw_df.columns)
    selected_columns = meta_columns + mapped_columns
    feature_columns_by_type[inspection_type] = selected_columns
    feature_rows.append(
        {
            "inspection_type": inspection_type,
            "meta_features": len(meta_columns),
            "mapped_inspection_features": len(mapped_columns),
            "total_model_features": len(selected_columns),
        }
    )

feature_summary = pd.DataFrame(feature_rows).set_index("inspection_type")
display(feature_summary)
logger.info("feature_mapping_verified=%s", feature_summary.to_dict(orient="index"))


,meta_features,mapped_inspection_features,total_model_features
inspection_type,,,
0,4,44,48
1,4,52,56
2,4,65,69
3,4,65,69
4,4,21,25


2026-08-25 15:34:24,807 | INFO | feature_mapping_verified={0: {'meta_features': 4, 'mapped_inspection_features': 44, 'total_model_features': 48}, 1: {'meta_features': 4, 'mapped_inspection_features': 52, 'total_model_features': 56}, 2: {'meta_features': 4, 'mapped_inspection_features': 65, 'total_model_features': 69}, 3: {'meta_features': 4, 'mapped_inspection_features': 65, 'total_model_features': 69}, 4: {'meta_features': 4, 'mapped_inspection_features': 21, 'total_model_features': 25}}


## 4. 동일한 시간순 Train/Validation/Test 분할

In [4]:
timestamp_group_sizes = raw_df.groupby(TIME_COLUMN, sort=True).size()
cumulative_rows = timestamp_group_sizes.cumsum().to_numpy()
timestamp_index = timestamp_group_sizes.index


def boundary_at(fraction: float):
    position = int(np.searchsorted(cumulative_rows, len(raw_df) * fraction, side="left"))
    return timestamp_index[position]


train_end_time = boundary_at(TRAIN_END_FRACTION)
validation_end_time = boundary_at(VALIDATION_END_FRACTION)
train_mask = raw_df[TIME_COLUMN] <= train_end_time
validation_mask = (
    (raw_df[TIME_COLUMN] > train_end_time)
    & (raw_df[TIME_COLUMN] <= validation_end_time)
)
test_mask = raw_df[TIME_COLUMN] > validation_end_time

train_df = raw_df.loc[train_mask]
validation_df = raw_df.loc[validation_mask]
test_df = raw_df.loc[test_mask]
assert train_df[TIME_COLUMN].max() < validation_df[TIME_COLUMN].min()
assert set(train_df[TIME_COLUMN]).isdisjoint(set(validation_df[TIME_COLUMN]))
assert validation_df[TIME_COLUMN].max() < test_df[TIME_COLUMN].min()
assert set(validation_df[TIME_COLUMN]).isdisjoint(set(test_df[TIME_COLUMN]))
assert int(train_mask.sum() + validation_mask.sum() + test_mask.sum()) == len(raw_df)

split_summary = pd.DataFrame(
    [
        {
            "split": "train",
            "rows": len(train_df),
            "positive_samples": int(train_df[TARGET].sum()),
            "positive_rate_pct": train_df[TARGET].mean() * 100,
            "timestamp_groups": train_df[TIME_COLUMN].nunique(),
            "start_time": train_df[TIME_COLUMN].min(),
            "end_time": train_df[TIME_COLUMN].max(),
        },
        {
            "split": "validation",
            "rows": len(validation_df),
            "positive_samples": int(validation_df[TARGET].sum()),
            "positive_rate_pct": validation_df[TARGET].mean() * 100,
            "timestamp_groups": validation_df[TIME_COLUMN].nunique(),
            "start_time": validation_df[TIME_COLUMN].min(),
            "end_time": validation_df[TIME_COLUMN].max(),
        },
        {
            "split": "test",
            "rows": len(test_df),
            "positive_samples": int(test_df[TARGET].sum()),
            "positive_rate_pct": test_df[TARGET].mean() * 100,
            "timestamp_groups": test_df[TIME_COLUMN].nunique(),
            "start_time": test_df[TIME_COLUMN].min(),
            "end_time": test_df[TIME_COLUMN].max(),
        },
    ]
).set_index("split")
display(split_summary)
evaluation_policy = pd.Series(
    {
        "model_selection_uses_test": False,
        "threshold_selected_on_test": False,
        "fixed_test_threshold": DECISION_THRESHOLD,
    },
    name="evaluation_policy",
)
display(evaluation_policy)
logger.info("split_summary=%s", split_summary.reset_index().to_dict(orient="records"))
logger.info("test_policy model_selection=False threshold=%.2f", DECISION_THRESHOLD)


,rows,positive_samples,positive_rate_pct,timestamp_groups,start_time,end_time
split,,,,,,
train,308196,1940,0.629470,29249,1970-06-23 03:58:55+00:00,1970-10-05 00:29:59+00:00
validation,44026,357,0.810884,3400,1970-10-05 00:30:30+00:00,1970-10-13 16:54:14+00:00
test,88052,2325,2.640485,7093,1970-10-13 16:54:52+00:00,1970-11-02 14:21:28+00:00


model_selection_uses_test     False
threshold_selected_on_test    False
fixed_test_threshold            0.5
Name: evaluation_policy, dtype: object

2026-08-25 15:34:25,183 | INFO | split_summary=[{'split': 'train', 'rows': 308196, 'positive_samples': 1940, 'positive_rate_pct': 0.6294695583330089, 'timestamp_groups': 29249, 'start_time': Timestamp('1970-06-23 03:58:55+0000', tz='UTC'), 'end_time': Timestamp('1970-10-05 00:29:59+0000', tz='UTC')}, {'split': 'validation', 'rows': 44026, 'positive_samples': 357, 'positive_rate_pct': 0.8108844773542907, 'timestamp_groups': 3400, 'start_time': Timestamp('1970-10-05 00:30:30+0000', tz='UTC'), 'end_time': Timestamp('1970-10-13 16:54:14+0000', tz='UTC')}, {'split': 'test', 'rows': 88052, 'positive_samples': 2325, 'positive_rate_pct': 2.640485167855358, 'timestamp_groups': 7093, 'start_time': Timestamp('1970-10-13 16:54:52+0000', tz='UTC'), 'end_time': Timestamp('1970-11-02 14:21:28+0000', tz='UTC')}]


2026-08-25 15:34:25,183 | INFO | test_policy model_selection=False threshold=0.50


## 5. 005 시간순 분할 데이터 저장

동일한 Train 0~70%, Validation 70~80%, Test 80~100% 분할을 `data/005_dataset/`에 CSV로 저장하고 재현용 메타데이터를 함께 기록한다.


In [5]:
SPLIT_DATA_DIR = REPO_ROOT / "data" / "005_dataset"
SPLIT_DATA_DIR.mkdir(parents=True, exist_ok=True)

split_frames = {
    "train": train_df,
    "validation": validation_df,
    "test": test_df,
}
split_file_rows = []

for split_name, split_frame in split_frames.items():
    split_path = SPLIT_DATA_DIR / f"{split_name}.csv"
    split_frame.to_csv(split_path, index=False)
    split_file_rows.append(
        {
            "split": split_name,
            "path": str(split_path.relative_to(REPO_ROOT)),
            "rows": int(len(split_frame)),
            "positive_samples": int(split_frame[TARGET].sum()),
            "positive_rate_pct": float(split_frame[TARGET].mean() * 100),
            "timestamp_groups": int(split_frame[TIME_COLUMN].nunique()),
            "start_time": split_frame[TIME_COLUMN].min().isoformat(),
            "end_time": split_frame[TIME_COLUMN].max().isoformat(),
            "sha256": sha256_file(split_path),
            "size_mb": split_path.stat().st_size / (1024 ** 2),
        }
    )

assert sum(row["rows"] for row in split_file_rows) == len(raw_df)
assert sum(row["positive_samples"] for row in split_file_rows) == int(raw_df[TARGET].sum())

split_metadata = {
    "experiment_id": EXPERIMENT_ID,
    "source_dataset_sha256": DATA_SHA256_BEFORE,
    "mapping_sha256": MAPPING_SHA256_BEFORE,
    "split_method": "timestamp_group_boundaries",
    "train_fraction": [0.0, TRAIN_END_FRACTION],
    "validation_fraction": [TRAIN_END_FRACTION, VALIDATION_END_FRACTION],
    "test_fraction": [VALIDATION_END_FRACTION, 1.0],
    "train_end_time": train_end_time.isoformat(),
    "validation_end_time": validation_end_time.isoformat(),
    "target_column": TARGET,
    "time_column": TIME_COLUMN,
    "type_column": TYPE_COLUMN,
    "files": split_file_rows,
}
SPLIT_METADATA_PATH = SPLIT_DATA_DIR / "split_metadata.json"
SPLIT_METADATA_PATH.write_text(
    json.dumps(split_metadata, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

split_artifact_summary = pd.DataFrame(split_file_rows).set_index("split")
display(split_artifact_summary)
logger.info(
    "split_dataset_saved directory=%s metadata=%s files=%s",
    SPLIT_DATA_DIR.relative_to(REPO_ROOT),
    SPLIT_METADATA_PATH.relative_to(REPO_ROOT),
    split_file_rows,
)


,path,rows,positive_samples,positive_rate_pct,timestamp_groups,start_time,end_time,sha256,size_mb
split,,,,,,,,,
train,data/005_dataset/train.csv,308196,1940,0.629470,29249,1970-06-23T03:58:55+00:00,1970-10-05T00:29:59+00:00,22494416452107ada36d817a8de1371d81edcb31936d74...,222.440481
validation,data/005_dataset/validation.csv,44026,357,0.810884,3400,1970-10-05T00:30:30+00:00,1970-10-13T16:54:14+00:00,839899671567694eae26a9412d7e190bc6845889015af3...,31.998715
test,data/005_dataset/test.csv,88052,2325,2.640485,7093,1970-10-13T16:54:52+00:00,1970-11-02T14:21:28+00:00,ab08cc604606bae8a197242937595d539d595d90c4213a...,63.699906


2026-08-25 15:34:36,938 | INFO | split_dataset_saved directory=data/005_dataset metadata=data/005_dataset/split_metadata.json files=[{'split': 'train', 'path': 'data/005_dataset/train.csv', 'rows': 308196, 'positive_samples': 1940, 'positive_rate_pct': 0.6294695583330089, 'timestamp_groups': 29249, 'start_time': '1970-06-23T03:58:55+00:00', 'end_time': '1970-10-05T00:29:59+00:00', 'sha256': '22494416452107ada36d817a8de1371d81edcb31936d74e4fd4f65ffa8b0b3d9', 'size_mb': 222.4404811859131}, {'split': 'validation', 'path': 'data/005_dataset/validation.csv', 'rows': 44026, 'positive_samples': 357, 'positive_rate_pct': 0.8108844773542907, 'timestamp_groups': 3400, 'start_time': '1970-10-05T00:30:30+00:00', 'end_time': '1970-10-13T16:54:14+00:00', 'sha256': '839899671567694eae26a9412d7e190bc6845889015af3a2c5db136736b7c4c0', 'size_mb': 31.9987154006958}, {'split': 'test', 'path': 'data/005_dataset/test.csv', 'rows': 88052, 'positive_samples': 2325, 'positive_rate_pct': 2.640485167855358, 'time

## 6. 동일한 평가 지표와 임계값 선택 함수

In [6]:
def evaluate_predictions(y_true, prediction, probability):
    y_true = np.asarray(y_true, dtype=np.int8)
    prediction = np.asarray(prediction, dtype=np.int8)
    probability = np.asarray(probability, dtype=np.float64)
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    has_both_classes = np.unique(y_true).size == 2
    has_positive = (tp + fn) > 0
    return {
        "rows": len(y_true),
        "positive_samples": int(y_true.sum()),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "accuracy": accuracy_score(y_true, prediction),
        "precision": precision_score(y_true, prediction, zero_division=0),
        "recall": recall_score(y_true, prediction, zero_division=0) if has_positive else np.nan,
        "false_call_reduction": tn / (tn + fp) if (tn + fp) else np.nan,
        "f1": f1_score(y_true, prediction, zero_division=0) if has_positive else np.nan,
        "roc_auc": roc_auc_score(y_true, probability) if has_both_classes else np.nan,
        "pr_auc": average_precision_score(y_true, probability) if has_both_classes else np.nan,
    }


def evaluate_probabilities(y_true, probability, threshold=DECISION_THRESHOLD):
    probability = np.asarray(probability, dtype=np.float64)
    prediction = (probability >= threshold).astype(np.int8)
    return evaluate_predictions(y_true, prediction, probability)


def select_threshold(y_true, probability, min_recall=MIN_RECALL):
    """Recall 제약을 만족하며 False Call Reduction이 최대인 threshold를 선택한다."""
    y_true = np.asarray(y_true, dtype=np.int8)
    probability = np.asarray(probability, dtype=np.float64)
    if np.unique(y_true).size != 2:
        raise ValueError("임계값 선택에는 positive와 negative가 모두 필요합니다.")

    order = np.argsort(-probability, kind="stable")
    sorted_probability = probability[order]
    sorted_target = y_true[order]
    cumulative_tp = np.cumsum(sorted_target == 1)
    cumulative_fp = np.cumsum(sorted_target == 0)
    group_ends = np.flatnonzero(
        np.r_[sorted_probability[:-1] != sorted_probability[1:], True]
    )

    thresholds = sorted_probability[group_ends]
    tp = cumulative_tp[group_ends]
    fp = cumulative_fp[group_ends]
    total_positive = int((y_true == 1).sum())
    total_negative = int((y_true == 0).sum())
    recall = tp / total_positive
    false_call_reduction = 1.0 - (fp / total_negative)
    feasible = np.flatnonzero(recall >= min_recall)
    if feasible.size == 0:
        raise RuntimeError(f"Recall {min_recall:.2%} 조건을 만족하는 threshold가 없습니다.")

    best_local = np.lexsort(
        (thresholds[feasible], recall[feasible], false_call_reduction[feasible])
    )[-1]
    best = feasible[best_local]
    selected_threshold = float(thresholds[best])
    metrics = evaluate_probabilities(y_true, probability, selected_threshold)
    return {"threshold": selected_threshold, "min_recall": min_recall, **metrics}


# 최적화 구현이 작은 합성 예제의 완전 탐색과 같은 결과인지 검증한다.
_test_y = np.array([1, 0, 1, 0, 1, 0], dtype=np.int8)
_test_probability = np.array([0.9, 0.8, 0.7, 0.6, 0.4, 0.2])
_optimized = select_threshold(_test_y, _test_probability, min_recall=2 / 3)
_reference_rows = []
for _threshold in np.unique(_test_probability):
    _metrics = evaluate_probabilities(_test_y, _test_probability, _threshold)
    if _metrics["recall"] >= 2 / 3:
        _reference_rows.append((_metrics["false_call_reduction"], _metrics["recall"], _threshold))
_reference = max(_reference_rows)
assert np.isclose(_optimized["threshold"], _reference[2])
logger.info("threshold_selector_unit_test=PASS")

def make_preprocessor(feature_columns):
    categorical = [column for column in meta_columns if column in feature_columns]
    continuous = [column for column in feature_columns if column not in categorical]
    return ColumnTransformer(
        transformers=[
            (
                "categorical",
                OneHotEncoder(handle_unknown="ignore", dtype=np.float32),
                categorical,
            ),
            ("continuous", "passthrough", continuous),
        ],
        sparse_threshold=1.0,
        verbose_feature_names_out=True,
    )


def evaluate_calibration_and_future(calibration_frame, calibration_probability, evaluation_frame, evaluation_probability, stage_name):
    global_selection = select_threshold(calibration_frame[TARGET], calibration_probability, min_recall=MIN_RECALL)
    type_thresholds = {}
    type_prediction = pd.Series(np.nan, index=evaluation_frame.index, dtype="float64")
    threshold_rows = [{"stage": stage_name, "scope": "global", **global_selection}]
    type_rows = []
    for inspection_type in inspection_types:
        type_calibration = calibration_frame.loc[calibration_frame[TYPE_COLUMN] == inspection_type]
        selection = select_threshold(type_calibration[TARGET], calibration_probability.loc[type_calibration.index], min_recall=MIN_RECALL)
        type_thresholds[inspection_type] = selection["threshold"]
        threshold_rows.append({"stage": stage_name, "scope": f"type_{inspection_type}", **selection})
        type_evaluation = evaluation_frame.loc[evaluation_frame[TYPE_COLUMN] == inspection_type]
        type_probability = evaluation_probability.loc[type_evaluation.index]
        prediction = (type_probability >= selection["threshold"]).astype("int8")
        type_prediction.loc[type_evaluation.index] = prediction
        metrics = evaluate_predictions(type_evaluation[TARGET], prediction, type_probability)
        type_rows.append({"stage": stage_name, "inspection_type": inspection_type, "threshold": selection["threshold"], **metrics})
    strategy_metrics = {
        "fixed_0.5": evaluate_probabilities(evaluation_frame[TARGET], evaluation_probability, DECISION_THRESHOLD),
        "global_threshold": evaluate_probabilities(evaluation_frame[TARGET], evaluation_probability, global_selection["threshold"]),
        "type_specific_thresholds": evaluate_predictions(evaluation_frame[TARGET], type_prediction, evaluation_probability),
    }
    metric_rows = [{"stage": stage_name, "strategy": strategy, **metrics} for strategy, metrics in strategy_metrics.items()]
    return {"global_selection": global_selection, "type_thresholds": type_thresholds, "threshold_rows": threshold_rows, "type_rows": type_rows, "metric_rows": metric_rows}

2026-08-25 15:34:36,963 | INFO | threshold_selector_unit_test=PASS


## 7. 동일한 3-Fold Expanding Walk-forward 구간

In [7]:
WALK_FORWARD_SPECS = [
    {
        "fold": "fold_1",
        "train_start": 0.00,
        "train_end": 0.30,
        "calibration_start": 0.30,
        "calibration_end": 0.40,
        "evaluation_start": 0.40,
        "evaluation_end": 0.50,
    },
    {
        "fold": "fold_2",
        "train_start": 0.00,
        "train_end": 0.40,
        "calibration_start": 0.40,
        "calibration_end": 0.50,
        "evaluation_start": 0.50,
        "evaluation_end": 0.60,
    },
    {
        "fold": "fold_3",
        "train_start": 0.00,
        "train_end": 0.50,
        "calibration_start": 0.50,
        "calibration_end": 0.60,
        "evaluation_start": 0.60,
        "evaluation_end": 0.70,
    },
]

walk_forward_boundaries = {
    fraction: boundary_at(fraction)
    for fraction in [0.30, 0.40, 0.50, 0.60, 0.70]
}
walk_forward_segments = {}
walk_forward_split_rows = []

for spec in WALK_FORWARD_SPECS:
    fold_name = spec["fold"]
    train_end = walk_forward_boundaries[spec["train_end"]]
    calibration_start = walk_forward_boundaries[spec["calibration_start"]]
    calibration_end = walk_forward_boundaries[spec["calibration_end"]]
    evaluation_start = walk_forward_boundaries[spec["evaluation_start"]]
    evaluation_end = walk_forward_boundaries[spec["evaluation_end"]]

    segments = {
        "train": raw_df.loc[raw_df[TIME_COLUMN] <= train_end],
        "calibration": raw_df.loc[
            (raw_df[TIME_COLUMN] > calibration_start)
            & (raw_df[TIME_COLUMN] <= calibration_end)
        ],
        "evaluation": raw_df.loc[
            (raw_df[TIME_COLUMN] > evaluation_start)
            & (raw_df[TIME_COLUMN] <= evaluation_end)
        ],
    }
    assert segments["train"][TIME_COLUMN].max() < segments["calibration"][TIME_COLUMN].min()
    assert segments["calibration"][TIME_COLUMN].max() < segments["evaluation"][TIME_COLUMN].min()
    assert set(segments["train"][TIME_COLUMN]).isdisjoint(segments["calibration"][TIME_COLUMN])
    assert set(segments["calibration"][TIME_COLUMN]).isdisjoint(segments["evaluation"][TIME_COLUMN])
    walk_forward_segments[fold_name] = segments

    for segment_name, frame in segments.items():
        walk_forward_split_rows.append(
            {
                "fold": fold_name,
                "segment": segment_name,
                "rows": len(frame),
                "positive_samples": int(frame[TARGET].sum()),
                "positive_rate_pct": frame[TARGET].mean() * 100,
                "timestamp_groups": frame[TIME_COLUMN].nunique(),
                "start_time": frame[TIME_COLUMN].min(),
                "end_time": frame[TIME_COLUMN].max(),
            }
        )

walk_forward_split_summary = pd.DataFrame(walk_forward_split_rows).set_index(
    ["fold", "segment"]
)
display(walk_forward_split_summary)
logger.info(
    "walk_forward_split_summary=%s",
    walk_forward_split_summary.reset_index().to_dict(orient="records"),
)


rows  positive_samples  positive_rate_pct  \
fold   segment                                                    
fold_1 train        132137              1223           0.925555   
       calibration   43979               200           0.454763   
       evaluation    44040               326           0.740236   
fold_2 train        176116              1423           0.807990   
       calibration   44040               326           0.740236   
       evaluation    44187               152           0.343993   
fold_3 train        220156              1749           0.794437   
       calibration   44187               152           0.343993   
       evaluation    43853                39           0.088933   

                    timestamp_groups                start_time  \
fold   segment                                                   
fold_1 train                   15230 1970-06-23 03:58:55+00:00   
       calibration              1251 1970-08-18 06:51:41+00:00   
       evaluation               5415 1970-08-21 23:33:55+00:00   
fold_2 train                   16481 1970-06-23 03:58:55+00:00   
       calibration              5415 1970-08-21 23:33:55+00:00   
       evaluation               4167 1970-09-15 06:47:13+00:00   
fold_3 train                   21896 1970-06-23 03:58:55+00:00   
       calibration              4167 1970-09-15 06:47:13+00:00   
       evaluation               3186 1970-09-28 05:11:13+00:00   

                                    end_time  
fold   segment                                
fold_1 train       1970-08-18 06:51:10+00:00  
       calibration 1970-08-21 23:32:59+00:00  
       evaluation  1970-09-15 06:46:33+00:00  
fold_2 train       1970-08-21 23:32:59+00:00  
       calibration 1970-09-15 06:46:33+00:00  
       evaluation  1970-09-28 05:10:37+00:00  
fold_3 train       1970-09-15 06:46:33+00:00  
       calibration 1970-09-28 05:10:37+00:00  
       evaluation  1970-10-05 00:29:59+00:00

2026-08-25 15:34:37,643 | INFO | walk_forward_split_summary=[{'fold': 'fold_1', 'segment': 'train', 'rows': 132137, 'positive_samples': 1223, 'positive_rate_pct': 0.9255545380930399, 'timestamp_groups': 15230, 'start_time': Timestamp('1970-06-23 03:58:55+0000', tz='UTC'), 'end_time': Timestamp('1970-08-18 06:51:10+0000', tz='UTC')}, {'fold': 'fold_1', 'segment': 'calibration', 'rows': 43979, 'positive_samples': 200, 'positive_rate_pct': 0.4547625002842266, 'timestamp_groups': 1251, 'start_time': Timestamp('1970-08-18 06:51:41+0000', tz='UTC'), 'end_time': Timestamp('1970-08-21 23:32:59+0000', tz='UTC')}, {'fold': 'fold_1', 'segment': 'evaluation', 'rows': 44040, 'positive_samples': 326, 'positive_rate_pct': 0.740236148955495, 'timestamp_groups': 5415, 'start_time': Timestamp('1970-08-21 23:33:55+0000', tz='UTC'), 'end_time': Timestamp('1970-09-15 06:46:33+0000', tz='UTC')}, {'fold': 'fold_2', 'segment': 'train', 'rows': 176116, 'positive_samples': 1423, 'positive_rate_pct': 0.807990188

## 8. 누적 시간 체크포인트별 Fold 모델 학습

In [8]:
ENSEMBLE_CHECKPOINTS = [0.30, 0.40, 0.50, 0.70]
FOLD_MEMBER_CHECKPOINTS = {"fold_1": [0.30], "fold_2": [0.30, 0.40], "fold_3": [0.30, 0.40, 0.50]}
FINAL_MEMBER_CHECKPOINTS = ENSEMBLE_CHECKPOINTS.copy()

prediction_targets = {}
for spec in WALK_FORWARD_SPECS:
    fold_name = spec["fold"]
    prediction_targets[f"{fold_name}_calibration"] = walk_forward_segments[fold_name]["calibration"]
    prediction_targets[f"{fold_name}_evaluation"] = walk_forward_segments[fold_name]["evaluation"]
prediction_targets["final_validation"] = validation_df
prediction_targets["final_test"] = test_df

checkpoint_predictions = {
    checkpoint: {
        target_name: pd.Series(np.nan, index=frame.index, dtype="float64")
        for target_name, frame in prediction_targets.items()
        if frame[TIME_COLUMN].min() > walk_forward_boundaries[checkpoint]
    }
    for checkpoint in ENSEMBLE_CHECKPOINTS
}
ensemble_members = {checkpoint: {} for checkpoint in ENSEMBLE_CHECKPOINTS}
ensemble_training_rows = []
for checkpoint in ENSEMBLE_CHECKPOINTS:
    checkpoint_train = raw_df.loc[raw_df[TIME_COLUMN] <= walk_forward_boundaries[checkpoint]]
    for inspection_type in inspection_types:
        feature_columns = feature_columns_by_type[inspection_type]
        type_train = checkpoint_train.loc[checkpoint_train[TYPE_COLUMN] == inspection_type]
        y_train = type_train[TARGET].astype("int8")
        assert y_train.nunique() == 2
        preprocessor = make_preprocessor(feature_columns)
        X_train = preprocessor.fit_transform(type_train[feature_columns])
        model = XGBClassifier(**XGB_PARAMS)
        model.fit(X_train, y_train, verbose=False)
        for target_name, probability_series in checkpoint_predictions[checkpoint].items():
            target_frame = prediction_targets[target_name]
            type_target = target_frame.loc[target_frame[TYPE_COLUMN] == inspection_type]
            X_target = preprocessor.transform(type_target[feature_columns])
            probability_series.loc[type_target.index] = model.predict_proba(X_target)[:, 1]
            del X_target
        ensemble_training_rows.append({
            "checkpoint": checkpoint, "inspection_type": inspection_type,
            "train_rows": len(type_train), "train_positive": int(y_train.sum()),
            "raw_features": len(feature_columns), "encoded_features": X_train.shape[1],
            "trees": model.get_booster().num_boosted_rounds(),
        })
        ensemble_members[checkpoint][inspection_type] = {
            "model": model,
            "preprocessor": preprocessor,
            "feature_columns": list(feature_columns),
        }
        logger.info("ensemble_member_fit_done checkpoint=%.2f type=%d rows=%d positive=%d", checkpoint, inspection_type, len(type_train), int(y_train.sum()))
        del X_train
        gc.collect()
for checkpoint, target_map in checkpoint_predictions.items():
    for target_name, probability in target_map.items():
        assert probability.notna().all(), (checkpoint, target_name)
ensemble_training_summary = pd.DataFrame(ensemble_training_rows).set_index(["checkpoint", "inspection_type"])
display(ensemble_training_summary)
logger.info("ensemble_members_trained=%d", len(ensemble_training_rows))

2026-08-25 15:34:38,074 | INFO | ensemble_member_fit_done checkpoint=0.30 type=0 rows=28277 positive=32


2026-08-25 15:34:38,554 | INFO | ensemble_member_fit_done checkpoint=0.30 type=1 rows=22698 positive=269


2026-08-25 15:34:39,343 | INFO | ensemble_member_fit_done checkpoint=0.30 type=2 rows=42288 positive=408


2026-08-25 15:34:40,163 | INFO | ensemble_member_fit_done checkpoint=0.30 type=3 rows=37264 positive=510


2026-08-25 15:34:40,256 | INFO | ensemble_member_fit_done checkpoint=0.30 type=4 rows=1610 positive=4


...

2026-08-25 15:34:48,948 | INFO | ensemble_member_fit_done checkpoint=0.70 type=2 rows=100470 positive=588


2026-08-25 15:34:50,091 | INFO | ensemble_member_fit_done checkpoint=0.70 type=3 rows=100740 positive=648


2026-08-25 15:34:50,182 | INFO | ensemble_member_fit_done checkpoint=0.70 type=4 rows=3813 positive=13


train_rows  train_positive  raw_features  \
checkpoint inspection_type                                             
0.3        0                     28277              32            48   
           1                     22698             269            56   
           2                     42288             408            69   
           3                     37264             510            69   
           4                      1610               4            25   
0.4        0                     36685              43            48   
           1                     26566             289            56   
           2                     58736             500            69   
           3                     51683             583            69   
           4                      2446               8            25   
0.5        0                     43181              93            48   
           1                     29184             475            56   
           2                     77700             549            69   
           3                     67320             622            69   
           4                      2771              10            25   
0.7        0                     64273             111            48   
           1                     38900             580            56   
           2                    100470             588            69   
           3                    100740             648            69   
           4                      3813              13            25   

                            encoded_features  trees  
checkpoint inspection_type                           
0.3        0                              80    400  
           1                             106    400  
           2                             114    400  
           3                             107    400  
           4                              47    400  
0.4        0                              82    400  
           1                             110    400  
           2                             114    400  
           3                             107    400  
           4                              47    400  
0.5        0                              84    400  
           1                             111    400  
           2                             115    400  
           3                             108    400  
           4                              50    400  
0.7        0                              88    400  
           1                             113    400  
           2                             117    400  
           3                             109    400  
           4                              53    400

2026-08-25 15:34:50,208 | INFO | ensemble_members_trained=20


## 9. Walk-forward 미래 Evaluation 결과

In [9]:
def mean_checkpoint_probability(checkpoints, target_name):
    probabilities = [checkpoint_predictions[checkpoint][target_name].to_numpy() for checkpoint in checkpoints]
    return pd.Series(np.mean(np.vstack(probabilities), axis=0), index=prediction_targets[target_name].index, dtype="float64")

walk_threshold_rows, walk_metric_rows, walk_type_rows = [], [], []
for spec in WALK_FORWARD_SPECS:
    fold_name = spec["fold"]
    members = FOLD_MEMBER_CHECKPOINTS[fold_name]
    result = evaluate_calibration_and_future(
        walk_forward_segments[fold_name]["calibration"], mean_checkpoint_probability(members, f"{fold_name}_calibration"),
        walk_forward_segments[fold_name]["evaluation"], mean_checkpoint_probability(members, f"{fold_name}_evaluation"), fold_name,
    )
    walk_threshold_rows.extend(result["threshold_rows"])
    walk_metric_rows.extend(result["metric_rows"])
    walk_type_rows.extend(result["type_rows"])
    logger.info("ensemble_walk_fold_done fold=%s checkpoints=%s", fold_name, members)

walk_forward_threshold_summary = pd.DataFrame(walk_threshold_rows).set_index(["stage", "scope"])
walk_forward_evaluation_metrics = pd.DataFrame(walk_metric_rows).set_index(["stage", "strategy"])
walk_forward_type_evaluation = pd.DataFrame(walk_type_rows).set_index(["stage", "inspection_type"])
walk_forward_strategy_summary = (
    walk_forward_evaluation_metrics.reset_index().groupby("strategy").agg(
        folds=("stage", "nunique"), mean_pr_auc=("pr_auc", "mean"),
        mean_recall=("recall", "mean"), min_recall=("recall", "min"),
        recall_99_folds=("recall", lambda values: int((values >= MIN_RECALL).sum())),
        mean_false_call_reduction=("false_call_reduction", "mean"),
        min_false_call_reduction=("false_call_reduction", "min"),
        total_tp=("tp", "sum"), total_fn=("fn", "sum"),
    )
)
display(walk_forward_threshold_summary[["threshold", "positive_samples", "recall", "false_call_reduction", "tp", "fn"]])
display(walk_forward_evaluation_metrics[["positive_samples", "pr_auc", "precision", "recall", "false_call_reduction", "f1", "tp", "fn", "fp", "tn"]])
display(walk_forward_type_evaluation[["threshold", "positive_samples", "pr_auc", "recall", "false_call_reduction", "tp", "fn"]])
display(walk_forward_strategy_summary)
logger.info("walk_forward_strategy_summary=%s", walk_forward_strategy_summary.to_dict(orient="index"))

2026-08-25 15:34:50,464 | INFO | ensemble_walk_fold_done fold=fold_1 checkpoints=[0.3]


2026-08-25 15:34:50,731 | INFO | ensemble_walk_fold_done fold=fold_2 checkpoints=[0.3, 0.4]


2026-08-25 15:34:50,987 | INFO | ensemble_walk_fold_done fold=fold_3 checkpoints=[0.3, 0.4, 0.5]


threshold  positive_samples    recall  false_call_reduction  \
stage  scope                                                                 
fold_1 global   0.000027               200  0.990000              0.012449   
       type_0   0.002342                11  1.000000              0.374539   
       type_1   0.001089                20  1.000000              0.146570   
       type_2   0.000025                92  1.000000              0.012900   
       type_3   0.000415                73  1.000000              0.211975   
       type_4   0.002461                 4  1.000000              0.000000   
fold_2 global   0.000488               326  0.990798              0.229812   
       type_0   0.000596                50  1.000000              0.330748   
       type_1   0.000350               186  0.994624              0.012747   
       type_2   0.000424                49  1.000000              0.255776   
       type_3   0.017858                39  1.000000              0.756764   
       type_4   0.002900                 2  1.000000              0.000000   
fold_3 global   0.000204               152  0.993421              0.204587   
       type_0   0.000468                14  1.000000              0.562479   
       type_1   0.001941                80  1.000000              0.589521   
       type_2   0.000146                32  1.000000              0.028384   
       type_3   0.000351                23  1.000000              0.334347   
       type_4   0.003162                 3  1.000000              0.000000   

                tp  fn  
stage  scope            
fold_1 global  198   2  
       type_0   11   0  
       type_1   20   0  
       type_2   92   0  
       type_3   73   0  
       type_4    4   0  
fold_2 global  323   3  
       type_0   50   0  
       type_1  185   1  
       type_2   49   0  
       type_3   39   0  
       type_4    2   0  
fold_3 global  151   1  
       type_0   14   0  
       type_1   80   0  
       type_2   32   0  
       type_3   23   0  
       type_4    3   0

positive_samples    pr_auc  precision  \
stage  strategy                                                          
fold_1 fixed_0.5                              326  0.134269   0.224490   
       global_threshold                       326  0.134269   0.007468   
       type_specific_thresholds               326  0.134269   0.008832   
fold_2 fixed_0.5                              152  0.028310   0.054545   
       global_threshold                       152  0.028310   0.004988   
       type_specific_thresholds               152  0.028310   0.006691   
fold_3 fixed_0.5                               39  0.037427   0.066667   
       global_threshold                        39  0.037427   0.001086   
       type_specific_thresholds                39  0.037427   0.001255   

                                   recall  false_call_reduction        f1  \
stage  strategy                                                             
fold_1 fixed_0.5                 0.134969              0.996523  0.168582   
       global_threshold          1.000000              0.008853  0.014825   
       type_specific_thresholds  0.981595              0.178478  0.017506   
fold_2 fixed_0.5                 0.039474              0.997638  0.045802   
       global_threshold          0.960526              0.338662  0.009925   
       type_specific_thresholds  0.822368              0.578563  0.013273   
fold_3 fixed_0.5                 0.025641              0.999680  0.037037   
       global_threshold          1.000000              0.181563  0.002170   
       type_specific_thresholds  1.000000              0.291551  0.002507   

                                  tp   fn     fp     tn  
stage  strategy                                          
fold_1 fixed_0.5                  44  282    152  43562  
       global_threshold          326    0  43327    387  
       type_specific_thresholds  320    6  35912   7802  
fold_2 fixed_0.5                   6  146    104  43931  
       global_threshold          146    6  29122  14913  
       type_specific_thresholds  125   27  18558  25477  
fold_3 fixed_0.5                   1   38     14  43800  
       global_threshold           39    0  35859   7955  
       type_specific_thresholds   39    0  31040  12774

threshold  positive_samples    pr_auc    recall  \
stage  inspection_type                                                    
fold_1 0                 0.002342                50  0.022766  0.940000   
       1                 0.001089               186  0.226506  0.983871   
       2                 0.000025                49  0.319577  1.000000   
       3                 0.000415                39  0.526485  1.000000   
       4                 0.002461                 2  0.006154  1.000000   
fold_2 0                 0.000596                14  0.004505  0.928571   
       1                 0.000350                80  0.261568  1.000000   
       2                 0.000424                32  0.031940  0.843750   
       3                 0.017858                23  0.002134  0.086957   
       4                 0.002900                 3  0.004298  1.000000   
fold_3 0                 0.000468                 4  0.006592  1.000000   
       1                 0.001941                25  0.033851  1.000000   
       2                 0.000146                 7  0.039505  1.000000   
       3                 0.000351                 3  0.169797  1.000000   
       4                 0.003162                 0       NaN       NaN   

                        false_call_reduction   tp  fn  
stage  inspection_type                                 
fold_1 0                            0.681663   47   3  
       1                            0.167763  183   3  
       2                            0.007296   49   0  
       3                            0.183485   39   0  
       4                            0.000000    2   0  
fold_2 0                            0.589678   13   1  
       1                            0.261582   80   0  
       2                            0.115146   27   5  
       3                            0.863347    2  21  
       4                            0.000000    3   0  
fold_3 0                            0.345038    4   0  
       1                            0.278706   25   0  
       2                            0.047331    7   0  
       3                            0.523520    3   0  
       4                            0.000000    0   0

,folds,mean_pr_auc,mean_recall,min_recall,recall_99_folds,mean_false_call_reduction,min_false_call_reduction,total_tp,total_fn
strategy,,,,,,,,,
fixed_0.5,3,0.066668,0.066695,0.025641,0,0.997947,0.996523,51,466
global_threshold,3,0.066668,0.986842,0.960526,2,0.176359,0.008853,511,6
type_specific_thresholds,3,0.066668,0.934655,0.822368,1,0.349530,0.178478,484,33


2026-08-25 15:34:51,004 | INFO | walk_forward_strategy_summary={'fixed_0.5': {'folds': 3, 'mean_pr_auc': 0.06666848491626425, 'mean_recall': 0.0666946783349754, 'min_recall': 0.02564102564102564, 'recall_99_folds': 0, 'mean_false_call_reduction': 0.9979471876094334, 'min_false_call_reduction': 0.9965228530905431, 'total_tp': 51, 'total_fn': 466}, 'global_threshold': {'folds': 3, 'mean_pr_auc': 0.06666848491626425, 'mean_recall': 0.9868421052631579, 'min_recall': 0.9605263157894737, 'recall_99_folds': 2, 'mean_false_call_reduction': 0.17635946579785103, 'min_false_call_reduction': 0.008852999039209407, 'total_tp': 511, 'total_fn': 6}, 'type_specific_thresholds': {'folds': 3, 'mean_pr_auc': 0.06666848491626425, 'mean_recall': 0.9346545043590572, 'min_recall': 0.8223684210526315, 'recall_99_folds': 1, 'mean_false_call_reduction': 0.3495304812388919, 'min_false_call_reduction': 0.1784782907077824, 'total_tp': 484, 'total_fn': 33}}


## 10. 최종 Validation 임계값 선택과 Walk-forward 모델의 Test 추론

In [10]:
validation_probability = mean_checkpoint_probability(FINAL_MEMBER_CHECKPOINTS, "final_validation")
test_probability = mean_checkpoint_probability(FINAL_MEMBER_CHECKPOINTS, "final_test")
model_summary = pd.Series({"ensemble_members": len(FINAL_MEMBER_CHECKPOINTS), "member_checkpoints": FINAL_MEMBER_CHECKPOINTS}, name="final_ensemble")

final_result = evaluate_calibration_and_future(validation_df, validation_probability, test_df, test_probability, "final_test")
global_threshold_selection = final_result["global_selection"]
thresholds_by_type = final_result["type_thresholds"]
threshold_summary = pd.DataFrame(final_result["threshold_rows"]).set_index(["stage", "scope"])
type_selected_test_metrics = pd.DataFrame(final_result["type_rows"]).set_index(["stage", "inspection_type"])
test_strategy_metrics = pd.DataFrame(final_result["metric_rows"]).set_index(["stage", "strategy"])
validation_fixed_metrics = pd.Series(evaluate_probabilities(validation_df[TARGET], validation_probability), name="validation_fixed_0.5")
type_validation_rows, type_test_rows = [], []
for inspection_type in inspection_types:
    type_validation = validation_df.loc[validation_df[TYPE_COLUMN] == inspection_type]
    type_test = test_df.loc[test_df[TYPE_COLUMN] == inspection_type]
    type_validation_rows.append({"inspection_type": inspection_type, **evaluate_probabilities(type_validation[TARGET], validation_probability.loc[type_validation.index])})
    type_test_rows.append({"inspection_type": inspection_type, **evaluate_probabilities(type_test[TARGET], test_probability.loc[type_test.index])})
type_validation_metrics = pd.DataFrame(type_validation_rows).set_index("inspection_type")
type_test_metrics = pd.DataFrame(type_test_rows).set_index("inspection_type")
display(model_summary)
display(threshold_summary[["threshold", "positive_samples", "recall", "false_call_reduction", "tp", "fn", "fp", "tn"]])
display(test_strategy_metrics[["pr_auc", "precision", "recall", "false_call_reduction", "f1", "tp", "fn", "fp", "tn"]])
display(type_selected_test_metrics[["threshold", "positive_samples", "pr_auc", "precision", "recall", "false_call_reduction", "tp", "fn", "fp", "tn"]])
display(type_validation_metrics)
display(type_test_metrics)
logger.info("validation_fixed_metrics=%s", validation_fixed_metrics.to_dict())
logger.info("test_strategy_metrics=%s", test_strategy_metrics.reset_index().to_dict(orient="records"))

ensemble_members                         4
member_checkpoints    [0.3, 0.4, 0.5, 0.7]
Name: final_ensemble, dtype: object

threshold  positive_samples    recall  \
stage      scope                                           
final_test global   0.000710               357  0.991597   
           type_0   0.000323                12  1.000000   
           type_1   0.001453               224  0.991071   
           type_2   0.000329                27  1.000000   
           type_3   0.002278                21  1.000000   
           type_4   0.003233                73  1.000000   

                   false_call_reduction   tp  fn     fp     tn  
stage      scope                                                
final_test global              0.611074  354   3  16984  26685  
           type_0              0.401371   12   0   7948   5329  
           type_1              0.522749  222   2   2958   3240  
           type_2              0.401177   27   0   4272   2862  
           type_3              0.834329   21   0   2689  13542  
           type_4              0.000000   73   0    829      0

pr_auc  precision    recall  \
stage      strategy                                                  
final_test fixed_0.5                 0.382545   0.719577  0.175484   
           global_threshold          0.382545   0.050436  0.939355   
           type_specific_thresholds  0.382545   0.046122  0.932903   

                                     false_call_reduction        f1    tp  \
stage      strategy                                                         
final_test fixed_0.5                             0.998145  0.282158   408   
           global_threshold                      0.520361  0.095733  2184   
           type_specific_thresholds              0.476734  0.087899  2169   

                                       fn     fp     tn  
stage      strategy                                      
final_test fixed_0.5                 1917    159  85568  
           global_threshold           141  41118  44609  
           type_specific_thresholds   156  44858  40869

threshold  positive_samples    pr_auc  precision  \
stage      inspection_type                                                     
final_test 0                 0.000323               195  0.050041   0.013536   
           1                 0.001453               774  0.559056   0.106601   
           2                 0.000329               731  0.549604   0.040078   
           3                 0.002278               612  0.238921   0.056961   
           4                 0.003233                13  0.017857   0.017857   

                              recall  false_call_reduction   tp  fn     fp  \
stage      inspection_type                                                   
final_test 0                0.861538              0.365516  168  27  12243   
           1                0.980620              0.450549  759  15   6361   
           2                0.960328              0.151322  702  29  16814   
           3                0.861111              0.745827  527  85   8725   
           4                1.000000              0.000000   13   0    715   

                               tn  
stage      inspection_type         
final_test 0                 7053  
           1                 5216  
           2                 2998  
           3                25602  
           4                    0

,rows,positive_samples,tn,fp,fn,tp,accuracy,precision,recall,false_call_reduction,f1,roc_auc,pr_auc
inspection_type,,,,,,,,,,,,,
0,13289,12,13277,0,12,0,0.999097,0.000000,0.000000,1.000000,0.000000,0.815022,0.003516
1,6422,224,6195,3,177,47,0.971971,0.940000,0.209821,0.999516,0.343066,0.965059,0.704424
2,7161,27,7132,2,18,9,0.997207,0.818182,0.333333,0.999720,0.473684,0.886664,0.363314
3,16252,21,16205,26,10,11,0.997785,0.297297,0.523810,0.998398,0.379310,0.969473,0.236705
4,902,73,829,0,73,0,0.919069,0.000000,0.000000,1.000000,0.000000,0.500000,0.080931


,rows,positive_samples,tn,fp,fn,tp,accuracy,precision,recall,false_call_reduction,f1,roc_auc,pr_auc
inspection_type,,,,,,,,,,,,,
0,19491,195,19296,0,195,0,0.989995,0.000000,0.000000,1.000000,0.000000,0.707807,0.050041
1,12351,774,11507,70,504,270,0.953526,0.794118,0.348837,0.993954,0.484740,0.905022,0.559056
2,20543,731,19805,7,670,61,0.967045,0.897059,0.083447,0.999647,0.152691,0.893032,0.549604
3,34939,612,34245,82,535,77,0.982341,0.484277,0.125817,0.997611,0.199741,0.873153,0.238921
4,728,13,715,0,13,0,0.982143,0.000000,0.000000,1.000000,0.000000,0.500000,0.017857


2026-08-25 15:34:51,616 | INFO | validation_fixed_metrics={'rows': 44026.0, 'positive_samples': 357.0, 'tn': 43638.0, 'fp': 31.0, 'fn': 290.0, 'tp': 67.0, 'accuracy': 0.9927088538590833, 'precision': 0.6836734693877551, 'recall': 0.1876750700280112, 'false_call_reduction': 0.9992901142687032, 'f1': 0.2945054945054945, 'roc_auc': 0.9377378513291323, 'pr_auc': 0.3826504981185914}


2026-08-25 15:34:51,617 | INFO | test_strategy_metrics=[{'stage': 'final_test', 'strategy': 'fixed_0.5', 'rows': 88052, 'positive_samples': 2325, 'tn': 85568, 'fp': 159, 'fn': 1917, 'tp': 408, 'accuracy': 0.9764230227592786, 'precision': 0.7195767195767195, 'recall': 0.17548387096774193, 'false_call_reduction': 0.9981452751175243, 'f1': 0.2821576763485477, 'roc_auc': 0.8839935273400394, 'pr_auc': 0.38254500187536195}, {'stage': 'final_test', 'strategy': 'global_threshold', 'rows': 88052, 'positive_samples': 2325, 'tn': 44609, 'fp': 41118, 'fn': 141, 'tp': 2184, 'accuracy': 0.531424612728842, 'precision': 0.0504364694471387, 'recall': 0.9393548387096774, 'false_call_reduction': 0.5203611464299462, 'f1': 0.09573278979551582, 'roc_auc': 0.8839935273400394, 'pr_auc': 0.38254500187536195}, {'stage': 'final_test', 'strategy': 'type_specific_thresholds', 'rows': 88052, 'positive_samples': 2325, 'tn': 40869, 'fp': 44858, 'fn': 156, 'tp': 2169, 'accuracy': 0.48877935765229635, 'precision': 0.04

## 11. 내부 Champion 모델 번들 저장과 재로딩 검증

체크포인트 4개 × Inspection Type 5개의 XGBoost 모델과 전처리기, 피처 목록, 앙상블 규칙 및 Validation에서 선택한 임계값을 동일 stem의 pickle 파일로 저장한다.


In [11]:
MODEL_PATH = REPO_ROOT / "models" / f"{EXPERIMENT_ID}.pkl"
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

model_bundle = {
    "schema_version": 2,
    "experiment_id": EXPERIMENT_ID,
    "model_family": "type_expert_expanding_checkpoint_ensemble",
    "random_state": RANDOM_STATE,
    "target_column": TARGET,
    "time_column": TIME_COLUMN,
    "type_column": TYPE_COLUMN,
    "record_id_column": RECORD_ID,
    "inspection_types": list(inspection_types),
    "xgb_params": dict(XGB_PARAMS),
    "ensemble_method": "equal_probability_mean",
    "ensemble_checkpoints": list(ENSEMBLE_CHECKPOINTS),
    "fold_member_checkpoints": dict(FOLD_MEMBER_CHECKPOINTS),
    "final_member_checkpoints": list(FINAL_MEMBER_CHECKPOINTS),
    "members": ensemble_members,
    "feature_columns_by_type": {
        key: list(value) for key, value in feature_columns_by_type.items()
    },
    "deduplicate_rows": False,
    "decision_threshold": float(global_threshold_selection["threshold"]),
    "global_threshold": float(global_threshold_selection["threshold"]),
    "type_thresholds": {
        key: float(value) for key, value in thresholds_by_type.items()
    },
    "threshold_min_recall": MIN_RECALL,
    "train_end_time": train_end_time.isoformat(),
    "validation_end_time": validation_end_time.isoformat(),
    "dataset_sha256": DATA_SHA256_BEFORE,
    "mapping_sha256": MAPPING_SHA256_BEFORE,
}

with MODEL_PATH.open("wb") as stream:
    pickle.dump(model_bundle, stream, protocol=pickle.HIGHEST_PROTOCOL)

with MODEL_PATH.open("rb") as stream:
    loaded_bundle = pickle.load(stream)

assert loaded_bundle["experiment_id"] == EXPERIMENT_ID
assert loaded_bundle["dataset_sha256"] == DATA_SHA256_BEFORE
assert loaded_bundle["mapping_sha256"] == MAPPING_SHA256_BEFORE
assert len(loaded_bundle["members"]) == len(FINAL_MEMBER_CHECKPOINTS)
assert sum(len(type_map) for type_map in loaded_bundle["members"].values()) == 20


def predict_from_saved_bundle(bundle, frame):
    checkpoint_probability = []
    for checkpoint in bundle["final_member_checkpoints"]:
        probability = pd.Series(np.nan, index=frame.index, dtype="float64")
        for inspection_type in bundle["inspection_types"]:
            member = bundle["members"][checkpoint][inspection_type]
            mask = frame[TYPE_COLUMN] == inspection_type
            X = member["preprocessor"].transform(
                frame.loc[mask, member["feature_columns"]]
            )
            probability.loc[mask] = member["model"].predict_proba(X)[:, 1]
            del X
        assert probability.notna().all()
        checkpoint_probability.append(probability.to_numpy())
    return pd.Series(
        np.mean(np.vstack(checkpoint_probability), axis=0),
        index=frame.index,
        dtype="float64",
    )


reloaded_validation_probability = predict_from_saved_bundle(loaded_bundle, validation_df)
assert np.allclose(
    reloaded_validation_probability.to_numpy(),
    validation_probability.to_numpy(),
    rtol=0.0,
    atol=1e-12,
)

MODEL_SHA256 = sha256_file(MODEL_PATH)
model_artifact_summary = pd.Series(
    {
        "model_path": str(MODEL_PATH.relative_to(REPO_ROOT)),
        "model_size_mb": MODEL_PATH.stat().st_size / (1024 ** 2),
        "model_sha256": MODEL_SHA256,
        "checkpoints": len(loaded_bundle["members"]),
        "types_per_checkpoint": len(loaded_bundle["inspection_types"]),
        "total_xgboost_models": sum(
            len(type_map) for type_map in loaded_bundle["members"].values()
        ),
        "reload_prediction_match": True,
    },
    name="saved_model_bundle",
)
display(model_artifact_summary)
logger.info("model_bundle_saved=%s", model_artifact_summary.to_dict())


model_path                 models/0825_peace_005_type_expert_fold_ensembl...
model_size_mb                                                       8.629811
model_sha256               e048d1f75f57060a9cbca2cae4763f8b8f4ba00f40977d...
checkpoints                                                                4
types_per_checkpoint                                                       5
total_xgboost_models                                                      20
reload_prediction_match                                                 True
Name: saved_model_bundle, dtype: object

2026-08-25 15:34:52,160 | INFO | model_bundle_saved={'model_path': 'models/0825_peace_005_type_expert_fold_ensemble.pkl', 'model_size_mb': 8.62981128692627, 'model_sha256': 'e048d1f75f57060a9cbca2cae4763f8b8f4ba00f40977d26e62b2059845b7023', 'checkpoints': 4, 'types_per_checkpoint': 5, 'total_xgboost_models': 20, 'reload_prediction_match': True}


## 12. 원본 무결성과 종료 확인


In [12]:
DATA_SHA256_AFTER = sha256_file(DATA_PATH)
MAPPING_SHA256_AFTER = sha256_file(MAPPING_PATH)
assert DATA_SHA256_AFTER == DATA_SHA256_BEFORE
assert MAPPING_SHA256_AFTER == MAPPING_SHA256_BEFORE
assert MODEL_PATH.exists() and MODEL_PATH.stat().st_size > 0
assert SPLIT_METADATA_PATH.exists()
assert all((SPLIT_DATA_DIR / f"{name}.csv").exists() for name in split_frames)
verification = pd.Series({
    "dataset_sha256_unchanged": True, "mapping_sha256_unchanged": True,
    "trained_model_units": len(FINAL_MEMBER_CHECKPOINTS) * len(inspection_types), "final_ensemble_members": len(FINAL_MEMBER_CHECKPOINTS),
    "test_evaluated_with_walk_forward_model": True,
    "fixed_threshold": DECISION_THRESHOLD,
    "global_threshold": global_threshold_selection["threshold"],
    "type_thresholds": thresholds_by_type,
    "split_dataset_directory": str(SPLIT_DATA_DIR.relative_to(REPO_ROOT)),
    "split_metadata_path": str(SPLIT_METADATA_PATH.relative_to(REPO_ROOT)),
    "split_rows_match_source": True,
    "model_path": str(MODEL_PATH.relative_to(REPO_ROOT)),
    "model_sha256": MODEL_SHA256,
    "model_reload_prediction_match": True,
    "log_file": f"docs/peace/{LOG_PATH.name}",
}, name="verification")
display(verification)
logger.info("source_integrity=PASS walk_forward_test_model=True")
logger.info("experiment_complete=%s", EXPERIMENT_ID)
for handler in logger.handlers:
    handler.flush()

dataset_sha256_unchanged                                                               True
mapping_sha256_unchanged                                                               True
trained_model_units                                                                      20
final_ensemble_members                                                                    4
test_evaluated_with_walk_forward_model                                                 True
fixed_threshold                                                                         0.5
global_threshold                                                                    0.00071
type_thresholds                           {0: 0.00032255006226478145, 1: 0.0014528487226...
split_dataset_directory                                                    data/005_dataset
split_metadata_path                                    data/005_dataset/split_metadata.json
split_rows_match_source                                                         

2026-08-25 15:34:52,351 | INFO | source_integrity=PASS walk_forward_test_model=True


2026-08-25 15:34:52,352 | INFO | experiment_complete=0825_peace_005_type_expert_fold_ensemble


## 13. 결론과 해석

- 최종 앙상블은 누적 체크포인트 4개 × Inspection Type 5개의 모델 20개로 구성된다.
- 모델·전처리기·피처·앙상블 규칙·Validation 임계값을 동일 stem의 pickle 번들로 저장했다.
- 저장 파일을 다시 로드한 Validation 확률이 학습 직후 확률과 완전히 일치하는지 검증했다.
- 이 파일은 내부 Champion 재현과 후속 추론용이며 Recall 99%를 아직 보장하는 최종 배포 모델은 아니다.
